#  Ames Housing Price Prediction: End-to-End ML Pipeline

This notebook demonstrates an end-to-end Machine Learning regression pipeline to predict residential property prices in Ames, Iowa using 80+ features.

### Workflow Overview:
1. **Exploratory Data Analysis (EDA)**: Target distribution, numeric correlation analysis, structural relationship visualizations.
2. **Pipeline Preprocessing**: Scikit-Learn `ColumnTransformer` with median imputation for numeric attributes, constant imputation for categorical attributes, standard feature scaling, and one-hot encoding.
3. **Model Benchmarking**: Comparing Multiple Linear Regression, Random Forest Regressor, and XGBoost Regressor.
4. **Hyperparameter Tuning**: 3-Fold Cross-Validated `GridSearchCV` on XGBoost Regressor.
5. **Feature Importance & Interpretation**: Analyzing top drivers of residential home prices.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['font.size'] = 11

## 1. Load Dataset & Exploratory Data Analysis

In [ ]:
# Load dataset from local data path
data_path = '../data/AmesHousing.csv' if os.path.exists('../data/AmesHousing.csv') else 'data/AmesHousing.csv'
df = pd.read_csv(data_path)

# Remove arbitrary identifiers
for col in ['Order', 'PID']:
    if col in df.columns:
        df = df.drop(columns=[col])

print(f"Ames Housing Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# Target Variable Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['SalePrice'], kde=True, color='skyblue', bins=30)
plt.title('Target Variable Distribution: SalePrice', fontsize=13, weight='bold')
plt.xlabel('Sale Price ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap of Top 10 Numeric Features
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=['int64', 'float64'])
top_corr_features = numeric_df.corr()['SalePrice'].abs().sort_values(ascending=False).head(10).index
sns.heatmap(numeric_df[top_corr_features].corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap: Top 10 Numeric Predictors with SalePrice', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

## 2. Preprocessing & Automated Feature Pipelines

In [ ]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

numerical_cols = list(X.select_dtypes(include=['int64', 'float64']).columns)
categorical_cols = list(X.select_dtypes(include=['object', 'category', 'string']).columns)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Processed Feature Space: {X_train_proc.shape[1]} attributes")

## 3. Model Training & Architecture Benchmarking

In [ ]:
models = {
    'Multiple Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost Regressor': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_test_proc)
    results.append({
        'Model': name,
        'R2 Score': r2_score(y_test, preds),
        'MAE ($)': mean_absolute_error(y_test, preds),
        'RMSE ($)': np.sqrt(mean_squared_error(y_test, preds))
    })

results_df = pd.DataFrame(results)
results_df

## 4. Hyperparameter Optimization (GridSearchCV on XGBoost)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_proc, y_train)

best_xgb = grid_search.best_estimator_
print(f"Best Hyperparameters: {grid_search.best_params_}")

tuned_preds = best_xgb.predict(X_test_proc)
print(f"Tuned XGBoost R²: {r2_score(y_test, tuned_preds):.4f}")
print(f"Tuned XGBoost MAE: ${mean_absolute_error(y_test, tuned_preds):,.2f}")
print(f"Tuned XGBoost RMSE: ${np.sqrt(mean_squared_error(y_test, tuned_preds)):,.2f}")

## 5. Visualizing Predictions & Top 10 Feature Importances

In [ ]:
# Actual vs Predicted Scatter Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, tuned_preds, alpha=0.6, color='crimson')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Perfect Fit')
plt.title('Actual vs. Predicted Sale Prices (Tuned XGBoost)', fontsize=13, weight='bold')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Extract and Plot Feature Importances
encoded_cat_cols = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
all_feature_names = list(numerical_cols) + list(encoded_cat_cols)

plt.figure(figsize=(10, 6))
feature_importances = pd.Series(best_xgb.feature_importances_, index=all_feature_names)
feature_importances.nlargest(10).plot(kind='barh', color='darkorange')
plt.title('Top 10 Influential Features for House Price Prediction (XGBoost)', fontsize=13, weight='bold')
plt.xlabel('Relative Importance Score')
plt.ylabel('Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()